# Sprint 2 - PixelPaw Back-End
### วิชา CP352301 Script Programming

**ทีม:** [ชื่อ Planner] (Planner/PM) · ศุภชัย คนเพียร (Coder) · [ชื่อ Debugger] (Debugger/QA)

**Repository:** `[แปะลิงก์ GitHub Repo]`

**Sprint:** Sprint 2 - Back-End App Dev (สัปดาห์ที่ 13)

หมายเหตุ: Sprint 2 รับงานทั้งหมดที่ **เกินขอบเขตของ Sprint 1** มาพัฒนาต่อ ได้แก่ Business Logic แบบ OOP, การเชื่อมต่อ External API และ File I/O (JSON Persistence) ส่วนหน้าจอ CLI พื้นฐานพัฒนาไว้แล้วในโฟลเดอร์ `Sprint1/code/`

---
## สารบัญ
1. [Planning - ขอบเขต Sprint 2](#1)
2. [Architecture ปัจจุบัน](#2)
3. [Source Code](#3)
4. [Demo / วิธีใช้งาน](#4)
5. [Test Report - Unit & Edge Case Testing](#5)
6. [Retrospective - Wow! & Whoops!](#6)
7. [แผนต่อยอดเป็น Sprint 3 (Full-Stack)](#7)
8. [บทบาทและ Self-assessment ตามเกณฑ์](#8)

In [ ]:
"""เซลล์ตั้งค่า - รันเซลล์นี้ก่อนเสมอ

โน้ตบุ๊กเก็บอยู่ที่ Sprint2/notebook/ ส่วนซอร์สโค้ดอยู่ที่ Sprint2/code/
เซลล์นี้จะย้ายไปทำงานที่โฟลเดอร์ code เพื่อให้ import แพ็กเกจ src ได้ถูกต้อง
"""
import os
import subprocess
import sys
from pathlib import Path

SPRINT_NAME = "Sprint2"


def find_code_dir(start):
    """ค้นหาโฟลเดอร์ code ให้เจอ ไม่ว่าจะเปิดโน้ตบุ๊กจากตำแหน่งใด"""
    candidates = [
        start.parent / "code",             # เปิดจาก Sprint2/notebook/ (ปกติ)
        start / "code",                    # เปิดจาก Sprint2/
        start / SPRINT_NAME / "code",      # เปิดจาก root ของ repo
        start,                             # อยู่ในโฟลเดอร์ code อยู่แล้ว
    ]
    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "tests").is_dir():
            return candidate.resolve()
    raise RuntimeError(f"ไม่พบโฟลเดอร์ code ของ {SPRINT_NAME}")


CODE_DIR = find_code_dir(Path.cwd())
os.chdir(CODE_DIR)

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))


def show_source(relative_path):
    """แสดงซอร์สโค้ดจริงจากไฟล์ เพื่อไม่ให้รายงานหลุดจากโค้ดใน repo"""
    print(f"# ===== {relative_path} =====")
    print((CODE_DIR / relative_path).read_text(encoding="utf-8"))


print("โฟลเดอร์โค้ด:", CODE_DIR)
print("ไฟล์ใน src/:", sorted(p.name for p in (CODE_DIR / "src").glob("*.py")))

<a id="1"></a>
## 1. Planning - ขอบเขต Sprint 2

**ภารกิจ Planner:**

**ขอบเขตระบบ (Scope):** เพิ่มชั้น Back-End ให้ PixelPaw โดยแปลงสถานะจาก `dict` ใน memory ให้เป็นอ็อบเจกต์ `Pet` (OOP), เชื่อมต่อ Gemini API เพื่อดึงข้อมูลสายพันธุ์แมว และบันทึกสถานะลงไฟล์ JSON เพื่อให้เล่นต่อจากครั้งก่อนได้

**งานที่อยู่ในสปรินต์นี้:**

| งาน | ไฟล์ | สถานะ |
|---|---|---|
| ตรรกะการเลี้ยง feed / play / rest แบบ OOP | `src/pet.py` | ทำแล้ว |
| แปลงสถานะเป็น dict และกลับ (serialization) | `src/pet.py` | ทำแล้ว |
| เชื่อม Gemini API ดึงข้อมูลสายพันธุ์ | `src/api_client.py` | ทำแล้ว |
| บันทึกและโหลดไฟล์ JSON | `src/app_functions.py` | ทำแล้ว |
| ลูปการเลี้ยงที่เชื่อมตรรกะจริง | `src/game.py` | ทำแล้ว |
| เมนูหลักเชื่อม API และไฟล์เซฟ | `src/app.py` | ทำแล้ว |
| Search / Filter / Sort รายการสายพันธุ์ | - | ยังไม่ทำ |
| Time Decay ตามเวลาจริง | - | ยังไม่ทำ |

**Definition of Done (DoD):**

| # | เงื่อนไข | ตรวจด้วยเทสต์ |
|---|---|---|
| 1 | `feed` / `play` / `rest` ปรับค่าสถานะถูกต้องตามสูตรที่ตกลงไว้ | `TestFeed`, `TestPlay`, `TestRest` |
| 2 | ค่าสถานะทุกตัวไม่เกิน 100 และไม่ต่ำกว่า 0 ไม่ว่าจะทำซ้ำกี่ครั้ง | `test_never_exceeds_max`, `test_energy_never_below_zero` |
| 3 | ค่าพลังงานที่เสียตอนเล่นต้องแปรผันตามคะแนนความดื้อของสายพันธุ์ | `test_costs_energy_based_on_stubbornness` |
| 4 | บันทึกแล้วโหลดกลับต้องได้สถานะเดิมครบทุกฟิลด์ (round-trip) | `test_round_trip_restores_stats` |
| 5 | ไม่มีไฟล์เซฟ หรือไฟล์เสียหาย ต้องคืน `None` ไม่ crash | `test_load_returns_none_when_file_missing`, `test_load_returns_none_when_file_is_corrupted` |
| 6 | เรียก API ไม่สำเร็จต้องคืน list ว่างพร้อมแจ้งเตือน ไม่ทำให้โปรแกรมหยุด | `test_fetch_new_pets_returns_empty_list_on_error` |
| 7 | ไม่ได้ตั้ง API Key ต้องแจ้งเตือนให้ชัดเจนก่อนเรียก API | `TestApiKeyGuard` |
| 8 | ชุดทดสอบทั้งหมดต้องรันได้โดยไม่ต้องต่ออินเทอร์เน็ตและไม่แตะไฟล์เซฟจริง | ทั้งโฟลเดอร์ `tests/` |

<a id="2"></a>
## 2. Architecture ปัจจุบัน (Back-End)

```
Terminal
   |
   v
main.py                        entry point + บังคับ UTF-8
   |
   v
src/app.py                     Application Layer (เมนูหลัก)
   |  +- start_new_game()          ดึงสายพันธุ์จาก API -> เลือก -> ตั้งชื่อ -> เซฟ
   |  +- continue_game()           โหลดเซฟแล้วเข้าลูปการเลี้ยง
   |
   +--> src/game.py            ลูปการเลี้ยง (เรียกตรรกะของ Pet + เซฟอัตโนมัติ)
   |
   +--> src/pet.py             Domain Layer (OOP)
   |       +- feed() / play() / rest()
   |       +- to_dict() / from_dict()
   |
   +--> src/app_functions.py   Data Layer
   |       +- save_game() / load_game()   <--> data/pet_state.json
   |       +- fetch_new_pets()            ---> src/api_client.py
   |
   +--> src/api_client.py      Integration Layer
           +- build_prompt() / strip_code_fence() / parse_response()
           +- get_cats_from_gemini()      ---> Gemini API
   |
   +--> src/key.py             อ่าน GEMINI_API_KEY จาก environment variable
```

**หลักการออกแบบที่ยึด:**
- **แยกชั้นชัดเจน** `pet.py` เป็นตรรกะบริสุทธิ์ ไม่รู้จักไฟล์และไม่รู้จัก API จึงทดสอบได้โดยไม่ต้องเตรียมอะไรเลย
- **แยกส่วนที่เทสต์ได้ออกจากส่วนที่ต้องต่อเน็ต** ใน `api_client.py` แยก `strip_code_fence()` และ `parse_response()` ออกมาเป็น static method ทำให้ทดสอบการแปลงข้อมูลได้โดยไม่ต้องเรียก API จริง
- **ไม่เก็บ API Key ไว้ในโค้ด** `key.py` อ่านจาก environment variable และมี `.env.example` เป็นตัวอย่าง
- **Lazy import** `from google import genai` ถูกย้ายไปไว้ในเมธอด ทำให้เทสต์รันได้แม้ยังไม่ติดตั้งไลบรารี

<a id="3"></a>
## 3. Source Code

เซลล์ด้านล่างอ่านซอร์สจากไฟล์จริงใน `src/` โดยตรง ทำให้รายงานตรงกับโค้ดใน repo เสมอ

In [ ]:
show_source("src/pet.py")

In [ ]:
show_source("src/api_client.py")

In [ ]:
show_source("src/app_functions.py")

In [ ]:
show_source("src/game.py")

In [ ]:
show_source("src/app.py")

<a id="4"></a>
## 4. Demo / วิธีใช้งาน

### รันจริงผ่าน Terminal

```bash
cd Sprint2/code
pip install -r requirements.txt
python main.py
```

ก่อนใช้เมนู 2 (เริ่มใหม่) ต้องตั้งค่า API Key ก่อน:

```powershell
$env:GEMINI_API_KEY = "คีย์ของคุณ"
```

### สาธิตในโน้ตบุ๊ก

เซลล์ถัดไปสาธิตแต่ละชั้นแยกกัน โดย **ไม่เรียก API จริงและไม่แตะไฟล์เซฟจริง**

In [ ]:
"""สาธิตชั้น Domain: ตรรกะการเลี้ยงของคลาส Pet"""
from src.pet import Pet

pet = Pet(
    name="มิว",
    breed="Siamese",
    country="ไทย",
    temperament="ฉลาด ขี้อ้อน",
    stubbornness_score=50,
    description="แมวไทยโบราณ",
)


def show(label):
    print(f"{label:<22} hunger={pet.hunger:<6} energy={pet.energy:<6} happiness={pet.happiness}")


show("เริ่มต้น")
pet.feed()
show("หลังให้อาหาร")
pet.play()
show("หลังเล่น")
pet.rest()
show("หลังนอนพัก")

print("\nพลังงานที่เสียตอนเล่น = 10 + (ความดื้อ / 10) =", 10 + pet.stubbornness_score / 10)

In [ ]:
"""สาธิตขอบเขตค่าสถานะ: ทำซ้ำหลายครั้งค่าต้องไม่หลุดช่วง 0-100"""
stress_pet = Pet("ทดสอบ", "Persian", "อิหร่าน", "ใจเย็น", 100, "-")

for _ in range(10):
    stress_pet.feed()
print("feed 10 ครั้ง  -> hunger =", stress_pet.hunger, "(ต้องไม่เกิน 100)")

for _ in range(10):
    stress_pet.play()
print("play 10 ครั้ง  -> energy =", stress_pet.energy, "(ต้องไม่ต่ำกว่า 0)")

In [ ]:
"""สาธิตชั้น Data: บันทึกและโหลดไฟล์ JSON แบบ round-trip

ใช้โฟลเดอร์ชั่วคราว จึงไม่กระทบไฟล์ data/pet_state.json ของจริง
"""
import tempfile
from pathlib import Path

from src import app_functions

temp_dir = Path(tempfile.mkdtemp())
original_dir, original_file = app_functions.SAVE_DIR, app_functions.SAVE_FILE
app_functions.SAVE_DIR = str(temp_dir / "data")
app_functions.SAVE_FILE = str(temp_dir / "data" / "pet_state.json")

try:
    app_functions.save_game(pet)
    print("--- เนื้อหาไฟล์ที่บันทึก ---")
    print(Path(app_functions.SAVE_FILE).read_text(encoding="utf-8"))

    restored = app_functions.load_game()
    print("โหลดกลับได้ชื่อ :", restored.name)
    print("สถานะตรงกับก่อนบันทึก :", restored.to_dict() == pet.to_dict())
finally:
    app_functions.SAVE_DIR, app_functions.SAVE_FILE = original_dir, original_file

In [ ]:
"""สาธิตชั้น Integration: แปลงผลลัพธ์จาก API โดยไม่ต้องเรียก API จริง"""
from src.api_client import APIClient

FAKE_RESPONSE = """```json
[
  {"breed": "Siamese", "country": "ไทย", "temperament": "ขี้อ้อน",
   "stubbornness_score": 45, "description": "แมวไทยโบราณ"}
]
```"""

parsed = APIClient.parse_response(FAKE_RESPONSE)
print("แปลง JSON สำเร็จ :", parsed[0]["breed"], "-", parsed[0]["country"])

try:
    APIClient.get_cats_from_gemini(api_key="")
except ValueError as error:
    print("ดักกรณีไม่ได้ตั้ง API Key ได้ :", error)

In [ ]:
"""สาธิตความทนทาน: API ล่ม โปรแกรมต้องไม่หยุด แต่คืน list ว่าง"""


def broken_api(*args, **kwargs):
    raise RuntimeError("เชื่อมต่อเซิร์ฟเวอร์ไม่ได้")


original_method = app_functions.APIClient.get_cats_from_gemini
app_functions.APIClient.get_cats_from_gemini = broken_api
try:
    print("ผลลัพธ์ที่ได้ :", app_functions.fetch_new_pets())
finally:
    app_functions.APIClient.get_cats_from_gemini = original_method

In [ ]:
"""รันเกมจริงแบบโต้ตอบ (ต้องตั้ง GEMINI_API_KEY ก่อนถ้าจะใช้เมนู 2)"""
# from src.app import main_menu
# main_menu()

<a id="5"></a>
## 5. Test Report - Unit & Edge Case Testing

**ภารกิจ Debugger:** ทดสอบทุกชั้นแยกกัน โดยออกแบบให้ชุดทดสอบทั้งหมด **ไม่ต้องต่ออินเทอร์เน็ตและไม่แตะไฟล์เซฟจริง** เพื่อให้ GitHub Actions รันผ่านโดยไม่ต้องใส่ Secret

| รายการทดสอบ | อินพุต / เงื่อนไข | ผลลัพธ์ที่คาดหวัง (Expected) | ผลการทดสอบจริง (Actual) | สถานะ |
|---|---|---|---|---|
| ค่าสถานะเริ่มต้น | สร้าง `Pet` ใหม่ | hunger/energy/happiness = 50 ทั้งหมด | ตรงตามคาด | **PASSED** |
| ให้อาหาร | `feed()` 1 ครั้ง | hunger +25, happiness +5 | 75 และ 55 | **PASSED** |
| ให้อาหารซ้ำจนล้น | `feed()` 10 ครั้ง | hunger ไม่เกิน 100 | หยุดที่ 100 | **PASSED** |
| เล่นกับแมว | `play()` 1 ครั้ง (ความดื้อ 50) | energy -15, happiness +20, hunger -10 | 35 / 70 / 40 | **PASSED** |
| เล่นซ้ำจนหมดแรง | `play()` 10 ครั้ง | energy ไม่ต่ำกว่า 0 | หยุดที่ 0 | **PASSED** |
| นอนพัก | `rest()` หลังเล่น | energy +35, hunger -5 | 70 และ 35 | **PASSED** |
| แปลงเป็น dict | `to_dict()` | มีครบ 9 ฟิลด์ | ครบถ้วน | **PASSED** |
| บันทึกแล้วโหลดกลับ | `save_game()` แล้ว `load_game()` | ได้สถานะเดิมทุกฟิลด์ | เท่ากันทุกฟิลด์ | **PASSED** |
| ไฟล์เซฟหาย | ยังไม่เคยบันทึก | คืน `None` ไม่ crash | คืน `None` | **PASSED** |
| ไฟล์เซฟเสียหาย | เขียน JSON ที่ผิดรูปแบบลงไฟล์ | คืน `None` ไม่ crash | ดัก `json.JSONDecodeError` ได้ | **PASSED** |
| ข้อมูลเซฟไม่ครบฟิลด์ | `from_dict({})` | ใช้ค่า default แทน ไม่ `KeyError` | ได้ชื่อ `น้องเหมียว` และค่า 50 | **PASSED** |
| API ล่ม | บังคับให้เมธอดโยน exception | คืน list ว่างพร้อมแจ้งเตือน | คืน `[]` และพิมพ์ข้อความ | **PASSED** |
| ไม่ได้ตั้ง API Key | `api_key=""` และ `None` | แจ้งเตือนให้ตั้งค่าก่อน | `ValueError` ระบุ `GEMINI_API_KEY` | **PASSED** |
| ผลลัพธ์ API มี markdown fence | ``` ครอบ JSON | ตัด fence ออกก่อนแปลง | `strip_code_fence()` ตัดได้ทั้งแบบ ```json และ ``` | **PASSED** |
| บันทึกภาษาไทย | ชื่อน้องแมวเป็นภาษาไทย | อ่านกลับได้ตรง ไม่เป็น `\uXXXX` | ใช้ `ensure_ascii=False` อ่านได้ปกติ | **PASSED** |

### รันชุดทดสอบอัตโนมัติจริง

In [ ]:
"""รันชุดทดสอบจริงด้วย pytest (ชุดเดียวกับที่ GitHub Actions ใช้ตรวจ)"""
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-v", "--no-header"],
    capture_output=True, text=True, encoding="utf-8", cwd=str(CODE_DIR),
)
print(result.stdout[-4000:])
print("exit code:", result.returncode)

### ตรวจมาตรฐานโค้ด (PEP 8)

ขั้นตอนเดียวกับที่ GitHub Actions ใช้ใน job `Lint & Test (Sprint2)`

In [ ]:
"""ตรวจมาตรฐานโค้ด PEP 8 ด้วย flake8 (ขั้นตอน Lint ของ CI)"""
result = subprocess.run(
    [sys.executable, "-m", "flake8", "."],
    capture_output=True, text=True, encoding="utf-8", cwd=str(CODE_DIR),
)
print(result.stdout or "flake8: ไม่พบข้อผิดพลาด (0 issues)")
print("exit code:", result.returncode)

<a id="6"></a>
## 6. Retrospective - Wow! & Whoops!

**Wow! (ส่วนที่ทำได้ดี):**
- คลาส `Pet` เป็นตรรกะบริสุทธิ์ ไม่ผูกกับไฟล์และไม่ผูกกับ API จึงเขียนเทสต์ได้ครบทุกเมธอดโดยไม่ต้องเตรียมสภาพแวดล้อมอะไรเลย
- แยก `strip_code_fence()` และ `parse_response()` ออกจากการเรียก API ทำให้ทดสอบส่วนที่พังบ่อยที่สุด (การแปลงข้อความเป็น JSON) ได้แบบออฟไลน์
- `load_game()` ดัก `json.JSONDecodeError`, `OSError`, `KeyError`, `TypeError` ครบ ทำให้ไฟล์เซฟที่เสียหายไม่ทำให้เกมพัง แค่เริ่มใหม่
- `from_dict()` ใช้ `.get()` พร้อมค่า default ทุกฟิลด์ ไฟล์เซฟรุ่นเก่าที่ยังไม่มีฟิลด์ใหม่จึงโหลดได้โดยไม่ error

**Whoops! (ปัญหาที่พบและแนวทางแก้ไข):**
- **เทสต์เดิมเรียกเมธอดที่ไม่มีอยู่จริง** `tests/test_pet.py` เดิมเรียก `get_cat_breeds_with_temperament()` ซึ่งไม่เคยถูกเขียนขึ้น แต่เพราะครอบ `try-except Exception` ไว้ทั้งก้อน เทสต์จึง "ผ่าน" ทั้งที่พัง แก้ไขด้วยการเขียนใหม่ให้ assert จริงและไม่กลืน exception
- **ไฟล์ทดสอบยิง API จริงตอนรัน CI** `test_api_connection.py` อยู่ที่ root และขึ้นต้นด้วย `test_` ทำให้ pytest เก็บไปรันแล้วเรียก API จริงซึ่งจะล้มเหลวบน CI เพราะไม่มี API Key แก้ไขโดยย้ายไปเป็น `tools/check_api_connection.py` ที่ตั้งใจไม่ตั้งชื่อขึ้นต้นด้วย `test_`
- **import พังเมื่อรันจากคนละโฟลเดอร์** โค้ดเดิมใช้ `sys.path.append()` แล้ว `from key import ...` ซึ่งใช้ได้เฉพาะตอนรันจากใน `src/` แก้ไขโดยทำ `src` ให้เป็นแพ็กเกจจริงและใช้ `from src.key import ...` พร้อมเพิ่ม `main.py` เป็น entry point
- **เทสต์เขียนทับไฟล์เซฟจริง** รอบแรก `save_game()` ในเทสต์เขียนทับ `data/pet_state.json` ของจริง แก้ไขด้วย `monkeypatch` เปลี่ยน `SAVE_FILE` ไปชี้ `tmp_path` ของ pytest
- **ต้องติดตั้ง google-genai ถึงจะ import โมดูลได้** ทำให้ CI ช้าและเทสต์ผูกกับไลบรารีที่ไม่ได้ใช้ในเทสต์ แก้ไขด้วยการย้าย `from google import genai` เข้าไปไว้ในเมธอด (lazy import) พร้อมข้อความแนะนำเมื่อยังไม่ได้ติดตั้ง

<a id="7"></a>
## 7. แผนต่อยอดเป็น Sprint 3 (Full-Stack)

เป้าหมายของ Sprint 3 คือเชื่อมหน้าจอของ Sprint 1 เข้ากับตรรกะของ Sprint 2 ให้เป็นแอปเดียวที่สมบูรณ์

| งาน | รายละเอียด | ปลายทาง |
|---|---|---|
| รวม Front-End เข้ากับ Back-End | นำ `validators.py` และ `ui.py` ของ Sprint 1 มาใช้กับเมนูของ Sprint 2 แทนการตรวจอินพุตแบบง่ายใน `app.py` | `src/app.py` |
| Search / Filter / Sort | ค้นหาสายพันธุ์ตามชื่อ กรองตามคะแนนความดื้อ และเรียงลำดับผลลัพธ์ | `src/app.py` |
| Time Decay | เก็บ `last_updated` ใน state แล้วลดค่าสถานะตามชั่วโมงที่ผ่านไปจริงตอนเปิดโปรแกรม | `src/time_decay.py` |
| ประวัติกิจกรรม | เก็บ `interaction_log` แบบ append-only ลงไฟล์เซฟด้วย | `src/pet.py` |
| Edge Cases เพิ่มเติม | ไฟล์เซฟจากเวอร์ชันเก่า ค่าสถานะติดลบจาก decay และการเล่นตอนพลังงานเป็น 0 | `tests/` |

### สิ่งที่ต้องระวังใน Sprint 3
- ตอนรวมโค้ดสองสปรินต์ ต้องไม่ทำให้เทสต์เดิมพัง ให้ย้ายทีละโมดูลแล้วรัน `pytest` ทุกครั้ง
- `time_decay` ต้องรับ timestamp เป็นพารามิเตอร์ ไม่เรียก `time.time()` ตรงๆ ในฟังก์ชันคำนวณ เพื่อให้ทดสอบได้

<a id="8"></a>
## 8. บทบาทและ Self-assessment ตามเกณฑ์การประเมิน

| บทบาท | ผู้รับผิดชอบ | งานหลักใน Sprint 2 |
|---|---|---|
| Planner / Team Leader | [ชื่อนักศึกษา] | กำหนดขอบเขต Back-End แยกงานที่ยกมาจาก Sprint 1 และนิยาม DoD 8 ข้อ |
| Coder | ศุภชัย คนเพียร | พัฒนา `pet.py` / `api_client.py` / `app_functions.py` / `game.py` / `app.py` |
| Debugger / QA | [ชื่อนักศึกษา] | ออกแบบเทสต์ออฟไลน์ 23 เคส ครอบคลุมตรรกะ ไฟล์ และการดักข้อผิดพลาดของ API |

### Self-assessment เทียบกับเกณฑ์ Role-Based Grading Rubric

| หัวข้อ | ระดับที่ประเมินตนเอง | เหตุผล |
|---|---|---|
| การออกแบบสถาปัตยกรรมและ OOP | [ระบุ] | แยก 4 ชั้นชัดเจน (Application / Domain / Data / Integration) คลาส `Pet` รวมทั้งสถานะและพฤติกรรมไว้ด้วยกัน |
| การประมวลผลข้อมูลและ Logic | [ระบุ] | ค่าพลังงานที่เสียแปรผันตามคะแนนความดื้อของสายพันธุ์ และทุกค่าถูกจำกัดช่วง 0-100 |
| การจัดการไฟล์และข้อมูล | [ระบุ] | `json.dump` ด้วย `ensure_ascii=False` รองรับภาษาไทย มี `os.path.exists` guard และดักไฟล์เสียหาย |
| การเชื่อมต่อ API และ Error Handling | [ระบุ] | ตรวจ API Key ก่อนเรียก ตัด markdown fence ก่อนแปลง JSON และคืน list ว่างเมื่อ API ล่ม |
| Debugger - การทดสอบ | [ระบุ] | 23 เคส รันแบบออฟไลน์ทั้งหมด ใช้ `monkeypatch` และ `tmp_path` จึงไม่แตะข้อมูลจริง |
| DevOps - CI/CD | [ระบุ] | job `Lint & Test (Sprint2)` ใน `.github/workflows/ci.yml` รัน flake8 และ pytest อัตโนมัติ |
| ความปลอดภัย | [ระบุ] | ไม่เก็บ API Key ในโค้ด อ่านจาก environment variable และใส่ `.env` ไว้ใน `.gitignore` |

### เอกสารอ้างอิง / Prompt ที่ใช้ในการพัฒนา
- Prompt ออกแบบคลาส `Pet` ให้แยกตรรกะออกจาก File I/O เพื่อให้ทดสอบได้
- Prompt ออกแบบการดัก Exception ของการเรียก API และการอ่านไฟล์เซฟที่เสียหาย
- Prompt แปลงเทสต์ที่ยิง API จริงให้เป็นเทสต์ออฟไลน์ที่รันบน CI ได้
- Prompt ตรวจสอบว่าโครงสร้าง import ของแพ็กเกจ `src` ถูกต้องตามมาตรฐาน Python